# Урок 11 - Протокол агент-до-агента (A2A)


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Що таке протокол A2A?

Протокол **Агент-до-агента (A2A)** є відкритим стандартом, який дозволяє агенгам штучного інтелекту спілкуватися, знаходити один одного та співпрацювати — навіть коли вони побудовані на різних фреймворках або розміщені різними сервісами.

Ключові поняття:

- **Виявлення** – Агенти публікують *картку агента*, яка описує їхні можливості, що полегшує іншим агентам (або оркестраторам) знайти потрібного фахівця для завдання.
- **Передача повідомлень** – Агенти обмінюються структурованими повідомленнями через спільний протокол, тож запит від одного агента може бути зрозумілий і виконаний іншим незалежно від його внутрішньої реалізації.
- **Життєвий цикл завдання** – A2A визначає стани, такі як *надіслано*, *виконується*, *завершено* і *не вдалося*, даючи оркестратору повну видимість того, як просувається делеговане завдання.

У цьому уроці ми імітуємо співпрацю у стилі A2A, зв'язавши три спеціалізовані туристичні агенти в робочий процес, де кожен агент вносить свою експертизу і передає результати наступному.


## Створення спеціалізованих туристичних агентів


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Співпраця кількох агентів через робочий процес

Ми поєднуємо три агенти в послідовний робочий процес, що відтворює передавання повідомлень A2A:

1. **CurrencyExchangeAgent** отримує запит користувача та надає рекомендації щодо обміну валюти.
2. **ActivityPlannerAgent** отримує збагачений контекст і додає рекомендації щодо заходів.
3. **TravelManagerAgent** синтезує обидва вхідні дані в підсумковий бриф щодо подорожі.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Розуміння A2A у виробництві

У виробничому середовищі протокол A2A відкриває потужні сценарії міжсервісної взаємодії:

| Capability | Description |
|---|---|
| **Міжфреймворкова взаємодія** | Агент, створений на одному фреймворку, може делегувати завдання агенту, створеному на будь-якому іншому фреймворку, сумісному з A2A, що дозволяє справжню міжорганізаційну взаємодію. |
| **Межі сервісів** | Агенти можуть існувати в окремих мікросервісах, регіонах хмари або навіть у різних організаціях, при цьому безперешкодно співпрацюючи. |
| **Динамічне виявлення** | Оркестратор може під час виконання звертатися до реєстру карток агента, щоб знайти найкращого спеціаліста для певного підзавдання. |
| **Потокова передача та push-повідомлення** | A2A підтримує Server-Sent Events (SSE) для оновлень прогресу в реальному часі та push-повідомлень для довготривалих завдань. |

Робочий процес, який ми побудували вище, є спрощеною, внутрішньопроцесною версією цього шаблону. У реальному
розгортанні кожен агент відкривав би HTTP-ендпойнт, публікував картку агента і спілкувався
через протокол A2A JSON-RPC.


## Підсумок

У цьому уроці ви дізналися:

1. **Що таке протокол A2A** — відкритий стандарт для виявлення між агентами, обміну повідомленнями,
   та управління завданнями.
2. **Як створювати спеціалізованих агентів** — агент обміну валют, агент планувальник активностей,
   та оркестратор управління подорожами.
3. **Як з'єднувати агентів у робочий процес** — використовуючи `WorkflowBuilder` для моделювання послідовної
   передачі повідомлень між агентами.
4. **Як A2A працює у виробництві** — забезпечуючи співпрацю між фреймворками та сервісами
   з динамічним виявленням і потоковими оновленнями.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Відмова від відповідальності:
Цей документ було перекладено за допомогою сервісу перекладу на основі штучного інтелекту [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, просимо врахувати, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ мовою оригіналу слід вважати авторитетним джерелом. Для критичної інформації рекомендується звернутися до послуг професійного перекладача. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
